# Import Libraries

In [2]:
!pip install geopy


   -------------------- ------------------- 1/2 [geopy]
   ---------------------------------------- 2/2 [geopy]



In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from geopy.distance import geodesic

# Load Dataset

In [5]:
df = pd.read_csv("DataCoSupplyChainDataset.csv", encoding="latin1")

print(df.shape)
df.head()

(180519, 53)


,Type,Days for shipping (real),Days for shipment (scheduled),Benefit per order,Sales per customer,Delivery Status,Late_delivery_risk,Category Id,Category Name,Customer City,...,Order Zipcode,Product Card Id,Product Category Id,Product Description,Product Image,Product Name,Product Price,Product Status,shipping date (DateOrders),Shipping Mode
0,DEBIT,3,4,91.250000,314.640015,Advance shipping,0,73,Sporting Goods,Caguas,...,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,2/3/2018 22:56,Standard Class
1,TRANSFER,5,4,-249.089996,311.359985,Late delivery,1,73,Sporting Goods,Caguas,...,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,1/18/2018 12:27,Standard Class
2,CASH,4,4,-247.779999,309.720001,Shipping on time,0,73,Sporting Goods,San Jose,...,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,1/17/2018 12:06,Standard Class
3,DEBIT,3,4,22.860001,304.809998,Advance shipping,0,73,Sporting Goods,Los Angeles,...,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,1/16/2018 11:45,Standard Class
4,PAYMENT,2,4,134.210007,298.250000,Advance shipping,0,73,Sporting Goods,Caguas,...,NaN,1360,73,NaN,http://images.acmesports.sports/Smart+watch,Smart watch,327.75,0,1/15/2018 11:24,Standard Class


# Standardize Column Names

In [6]:
df.columns = [col.lower().replace(" ", "_") for col in df.columns]

# Parse Dates

In [8]:
date_cols = [c for c in df.columns if 'date' in c]

for col in date_cols:
    df[col] = pd.to_datetime(df[col], errors='coerce')

print(date_cols)

['order_date_(dateorders)', 'shipping_date_(dateorders)']


# Remove Missing Target

In [10]:
print([c for c in df.columns if 'date' in c])

['order_date_(dateorders)', 'shipping_date_(dateorders)']


In [11]:
order_col = [c for c in df.columns if 'order' in c and 'date' in c][0]
ship_col = [c for c in df.columns if 'shipping' in c and 'date' in c][0]

df['shipping_duration_days'] = (
    df[ship_col] - df[order_col]
).dt.days

In [12]:
df = df.dropna(subset=['shipping_duration_days'])

In [13]:
df['shipping_duration_days'].describe()

count    180519.000000
mean          3.471856
std           1.670471
min           0.000000
25%           2.000000
50%           3.000000
75%           5.000000
max           6.000000
Name: shipping_duration_days, dtype: float64

In [14]:
df = df[df['shipping_duration_days'] >= 0]

In [15]:
df.columns = df.columns.str.lower().str.replace(" ","_")

# Missing Value Handling
## Numeric Imputation + Missing Flags

In [16]:
numeric_cols = ['product_weight','order_value','discount_offered']

for col in numeric_cols:
    if col in df.columns:
        df[f'{col}_was_missing'] = df[col].isnull().astype(int)
        df[col] = df.groupby('product_category')[col].transform(
            lambda x: x.fillna(x.median())
        )
        df[col] = df[col].fillna(df[col].median())

### Categorical Imputation

In [17]:
cat_cols = ['payment_mode','product_category']

for col in cat_cols:
    if col in df.columns:
        df[col] = df[col].fillna("Unknown")

# Remove Invalid Records

In [20]:
initial = len(df)

# impossible dates
df = df[df['shipping_date_(dateorders)'] >= df['order_date_(dateorders)']]

# remove outliers (95 percentile)
p95 = df['shipping_duration_days'].quantile(0.95)
df = df[df['shipping_duration_days'] <= p95]

# invalid financials
df = df[df['order_item_total'] > 0]

# remove duplicates
df = df.drop_duplicates(subset=['order_id'])

print("Removed:", initial - len(df))
print("Final:", len(df))

Removed: 114767
Final: 65752


In [19]:
print(df.columns.tolist())

['type', 'days_for_shipping_(real)', 'days_for_shipment_(scheduled)', 'benefit_per_order', 'sales_per_customer', 'delivery_status', 'late_delivery_risk', 'category_id', 'category_name', 'customer_city', 'customer_country', 'customer_email', 'customer_fname', 'customer_id', 'customer_lname', 'customer_password', 'customer_segment', 'customer_state', 'customer_street', 'customer_zipcode', 'department_id', 'department_name', 'latitude', 'longitude', 'market', 'order_city', 'order_country', 'order_customer_id', 'order_date_(dateorders)', 'order_id', 'order_item_cardprod_id', 'order_item_discount', 'order_item_discount_rate', 'order_item_id', 'order_item_product_price', 'order_item_profit_ratio', 'order_item_quantity', 'sales', 'order_item_total', 'order_profit_per_order', 'order_region', 'order_state', 'order_status', 'order_zipcode', 'product_card_id', 'product_category_id', 'product_description', 'product_image', 'product_name', 'product_price', 'product_status', 'shipping_date_(dateorde

# Categorical Standardization

In [21]:
df['shipping_mode'] = df['shipping_mode'].str.lower().str.strip()

mapping = {
    'air':'Air',
    'flight':'Air',
    'road':'Road',
    'ground':'Road',
    'ship':'Ship',
    'multiple':'Multiple'
}

df['shipping_mode'] = df['shipping_mode'].map(mapping)

# TEMPORAL FEATURE ENGINEERING


In [23]:
# use correct order date column
order_col = 'order_date_(dateorders)'

# ensure datetime
df[order_col] = pd.to_datetime(df[order_col], errors='coerce')

# ---- Temporal Features ----
df['order_hour'] = df[order_col].dt.hour
df['order_day_of_week'] = df[order_col].dt.dayofweek
df['order_day_of_month'] = df[order_col].dt.day
df['order_month'] = df[order_col].dt.month
df['order_quarter'] = df[order_col].dt.quarter

# weekend flag
df['is_weekend'] = (df['order_day_of_week'] >= 5).astype(int)

# festival season
df['is_festival_season'] = df['order_month'].isin([10,11,12,1]).astype(int)

# holiday flag
holidays = [(1,26),(8,15),(10,2),(12,25)]

def is_holiday(d):
    return int((d.month,d.day) in holidays)

df['is_holiday'] = df[order_col].apply(is_holiday)

print(df[['order_hour','order_day_of_week','is_weekend']].head())

   order_hour  order_day_of_week  is_weekend
0          22                  2           0
1          12                  5           1
2          12                  5           1
3          11                  5           1
4          11                  5           1


# GEOSPATIAL FEATURES

In [25]:
# origin and destination zones using states
df['origin_zone'] = df['order_state'].astype('category').cat.codes
df['destination_zone'] = df['customer_state'].astype('category').cat.codes

# route id
df['route_id'] = (
    df['origin_zone'].astype(str) + "_" +
    df['destination_zone'].astype(str)
)

# same zone delivery
df['same_zone_delivery'] = (
    df['origin_zone'] == df['destination_zone']
).astype(int)

# simple distance proxy (state difference)
df['distance_proxy'] = (
    abs(df['origin_zone'] - df['destination_zone'])
)

##  KMeans Zones

In [27]:
df['order_value_per_unit'] = (
    df['order_item_total'] /
    (df['order_item_quantity'] + 1)
)

df['discount_intensity'] = (
    df['order_item_discount'] /
    (df['order_item_total'] + 1)
)

df['profit_to_value_ratio'] = (
    df['order_item_profit_ratio'] /
    (df['order_item_total'] + 1)
)

## Distance Proxy

In [29]:
df['origin_zone'] = df['order_state'].astype('category').cat.codes
df['destination_zone'] = df['customer_state'].astype('category').cat.codes

df['route_id'] = (
    df['origin_zone'].astype(str) + "_" +
    df['destination_zone'].astype(str)
)

df['same_zone_delivery'] = (
    df['origin_zone'] == df['destination_zone']
).astype(int)

df['distance_proxy'] = abs(
    df['origin_zone'] - df['destination_zone']
)

In [30]:
df = df.sort_values('order_date_(dateorders)')

df['zone_pair_avg_lead_time_7d'] = (
    df.groupby('route_id')['shipping_duration_days']
      .transform(lambda x: x.rolling(7, min_periods=1).mean())
)

df['zone_pair_std_lead_time_7d'] = (
    df.groupby('route_id')['shipping_duration_days']
      .transform(lambda x: x.rolling(7, min_periods=1).std())
)

df['daily_order_volume'] = (
    df.groupby(df['order_date_(dateorders)'].dt.date)['order_id']
      .transform('count')
)

## Delivery Dynamics Features

In [31]:


df['order_value_per_unit'] = (
    df['order_item_total'] /
    (df['order_item_quantity'] + 1)
)

df['delivery_speed_ratio'] = (
    df['distance_proxy'] /
    (df['shipping_duration_days'] + 0.1)
)

df['capacity_proxy'] = (
    df['order_item_total'] /
    (df['order_item_quantity'] + 0.1)
)

df['profit_to_value_ratio'] = (
    df['order_item_profit_ratio'] /
    (df['order_item_total'] + 1)
)

df['discount_intensity'] = (
    df['order_item_discount'] /
    (df['order_item_total'] + 1)
)

print("Delivery dynamics features created")

Delivery dynamics features created


## Rolling features

In [32]:

df = df.sort_values('order_date_(dateorders)')


df['zone_pair_avg_lead_time_7d'] = (
    df.groupby('route_id')['shipping_duration_days']
      .transform(lambda x: x.rolling(7, min_periods=1).mean())
)

df['zone_pair_std_lead_time_7d'] = (
    df.groupby('route_id')['shipping_duration_days']
      .transform(lambda x: x.rolling(7, min_periods=1).std())
)


df['daily_order_volume'] = (
    df.groupby(df['order_date_(dateorders)'].dt.date)['order_id']
      .transform('count')
)

print("Rolling features created")

Rolling features created


## One Hot Encoding

In [33]:
df = pd.get_dummies(
    df,
    columns=['shipping_mode','order_status','customer_segment'],
    drop_first=True
)

print("Encoding complete")

Encoding complete


## Drop Leakage/Unused Columns

In [34]:
drop_cols = [
    'order_date_(dateorders)',
    'shipping_date_(dateorders)',
    'customer_email',
    'customer_fname',
    'customer_lname',
    'customer_password',
    'product_image'
]

df = df.drop(columns=[c for c in drop_cols if c in df.columns])

print("Dropped unnecessary columns")

Dropped unnecessary columns


## Features Matrix

In [35]:
target = 'shipping_duration_days'

X = df.drop(target, axis=1)
y = df[target]

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (65752, 74)
y shape: (65752,)


## Train / Val / Test Split (80/10/10)

In [36]:
from sklearn.model_selection import train_test_split

X_train_temp, X_test, y_train_temp, y_test = train_test_split(
    X, y,
    test_size=0.10,
    random_state=42,
    stratify=pd.cut(y, bins=10)
)

X_train, X_val, y_train, y_val = train_test_split(
    X_train_temp,
    y_train_temp,
    test_size=0.10/0.90,
    random_state=42,
    stratify=pd.cut(y_train_temp, bins=10)
)

print(X_train.shape, X_val.shape, X_test.shape)

(52600, 74) (6576, 74) (6576, 74)


## Features Scaling

In [40]:
from sklearn.preprocessing import StandardScaler


num_cols = X_train.select_dtypes(include=np.number).columns


zero_var_cols = [c for c in num_cols if X_train[c].std() == 0]

X_train = X_train.drop(columns=zero_var_cols)
X_val = X_val.drop(columns=zero_var_cols)
X_test = X_test.drop(columns=zero_var_cols)


num_cols = X_train.select_dtypes(include=np.number).columns


X_train[num_cols] = X_train[num_cols].fillna(0)
X_val[num_cols] = X_val[num_cols].fillna(0)
X_test[num_cols] = X_test[num_cols].fillna(0)


scaler = StandardScaler()

X_train[num_cols] = scaler.fit_transform(X_train[num_cols])
X_val[num_cols] = scaler.transform(X_val[num_cols])
X_test[num_cols] = scaler.transform(X_test[num_cols])

print("Scaling complete ")

Scaling complete 


# Files

In [41]:
X_train.to_csv("X_train.csv", index=False)
X_val.to_csv("X_val.csv", index=False)
X_test.to_csv("X_test.csv", index=False)

y_train.to_csv("y_train.csv", index=False)
y_val.to_csv("y_val.csv", index=False)
y_test.to_csv("y_test.csv", index=False)

print("All files saved successfully")

All files saved successfully
